# KEN3238 — Mixed Latin/Devanagari Digit Classification

Each image is a handwritten digit in **Latin (`0–9`)** or **Devanagari (`०–९`)**.
We predict the digit value `0–9` regardless of script.

Pipeline:

1. Load `train/train/` via `ImageFolder` (folder name ↔ digit label).
2. Visualize a handful of samples per class so we know what Latin vs. Devanagari look like.
3. Train a small CNN on 32×32 grayscale crops with mild augmentation.
4. Predict on `test/test/` in the order given by `test.csv`, write `submission.csv`.

This notebook is a narrative companion to `train.py`. Running every cell end-to-end reproduces the same `submission.csv`.

## Setup

In [ ]:
import csv
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

SEED = 42
IMG_SIZE = 32
BATCH_SIZE = 128
VAL_FRACTION = 0.10
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4

DATA_DIR = Path.cwd()
TRAIN_DIR = DATA_DIR / "train" / "train"
TEST_DIR = DATA_DIR / "test" / "test"
CKPT_PATH = DATA_DIR / "best_model.pt"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"
SUBMISSION_PATH = DATA_DIR / "submission.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"device: {device}")

## Data loading

`ImageFolder` gives us `(PIL image, label)` pairs, where `label` is the alphabetical index of the folder name. Folders are `'0'..'9'` so labels are `0..9` directly — no remapping needed. We split 90/10 into train/val with a seeded generator so the split is reproducible.

In [ ]:
normalize = transforms.Normalize((0.5,), (0.5,))
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    normalize,
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        with Image.open(path) as im:
            return self.transform(im.convert("L")), label

full = datasets.ImageFolder(str(TRAIN_DIR))
n_val = int(len(full) * VAL_FRACTION)
n_train = len(full) - n_val
gen = torch.Generator().manual_seed(SEED)
train_sub, val_sub = random_split(full, [n_train, n_val], generator=gen)
train_ds = TransformSubset(train_sub, train_tf)
val_ds = TransformSubset(val_sub, eval_tf)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

print(f"class_to_idx: {full.class_to_idx}")
print(f"total: {len(full)}  train: {len(train_ds)}  val: {len(val_ds)}")
counts = Counter(label for _, label in full.samples)
print(f"per-class counts: {dict(sorted(counts.items()))}")

## Sample visualizations

Each class folder contains both Latin and Devanagari examples of that digit. Plotting a grid makes the script ambiguity obvious — and why the classifier has to learn both forms of "1", both forms of "2", etc.

In [ ]:
samples_by_class = {i: [] for i in range(10)}
for path, label in full.samples:
    if len(samples_by_class[label]) < 8:
        samples_by_class[label].append(path)
    if all(len(v) == 8 for v in samples_by_class.values()):
        break

fig, axes = plt.subplots(10, 8, figsize=(10, 12))
for digit in range(10):
    for j, p in enumerate(samples_by_class[digit]):
        ax = axes[digit][j]
        ax.imshow(Image.open(p).convert("L"), cmap="gray")
        ax.axis("off")
        if j == 0:
            ax.set_ylabel(str(digit), rotation=0, labelpad=20, fontsize=12)
fig.suptitle("8 random training samples per class (0–9)", y=0.92)
plt.show()

## Model

A small CNN: three `Conv→ReLU` blocks with two `MaxPool`s, then two dense layers. Dropout at 0.25 after the first pool and 0.5 before the output. ~1.1M parameters — trivial to train on 15k images.

In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.drop1 = nn.Dropout(0.25)
        self.drop2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.drop1(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)

model = Net().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

## Training

AdamW + cosine-annealing LR schedule over 10 epochs. We track both train and val accuracy and save the best-val checkpoint to `best_model.pt`.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0; correct = 0; total = 0
    sum_loss = nn.CrossEntropyLoss(reduction="sum")
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += sum_loss(logits, y).item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

history = []
best_val = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0; tr_correct = 0; tr_total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * y.size(0)
        tr_correct += (logits.argmax(1) == y).sum().item()
        tr_total += y.size(0)
    tr_loss /= tr_total; tr_acc = tr_correct / tr_total
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step()
    history.append((epoch, tr_loss, tr_acc, val_loss, val_acc))
    print(f"epoch {epoch:02d}/{EPOCHS}  train acc={tr_acc:.4f}  val acc={val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), CKPT_PATH)
print(f"best val accuracy: {best_val:.4f}")

In [ ]:
hist = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist[:, 0], hist[:, 1], label="train")
ax[0].plot(hist[:, 0], hist[:, 3], label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Loss")
ax[1].plot(hist[:, 0], hist[:, 2], label="train")
ax[1].plot(hist[:, 0], hist[:, 4], label="val")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend(); ax[1].set_title("Accuracy")
plt.show()

## Test inference and submission

We load the best checkpoint, run it over every image listed in `test.csv`, and write `submission.csv` in the exact order of `test.csv`. Final asserts confirm the row count and header match the sample file.

In [ ]:
class TestDataset(Dataset):
    def __init__(self, ids, img_dir, transform):
        self.ids = ids; self.img_dir = img_dir; self.transform = transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        with Image.open(self.img_dir / f"{img_id}.png") as im:
            return self.transform(im.convert("L")), img_id

model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

test_ids = pd.read_csv(TEST_CSV)["Id"].tolist()
test_ds = TestDataset(test_ids, TEST_DIR, eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

predictions = {}
with torch.no_grad():
    for x, ids in test_loader:
        x = x.to(device)
        preds = model(x).argmax(1).cpu().tolist()
        for img_id, pred in zip(ids.tolist(), preds):
            predictions[img_id] = pred

with open(SAMPLE_SUB, newline="") as f:
    sample_header = next(csv.reader(f))
assert sample_header == ["Id", "Category"], sample_header

with open(SUBMISSION_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Id", "Category"])
    for img_id in test_ids:
        writer.writerow([img_id, predictions[img_id]])

assert len(predictions) == len(test_ids) == 3000
print(f"wrote {SUBMISSION_PATH} ({len(test_ids)} rows)")

In [ ]:
# Peek at a handful of test predictions visually.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img_id in zip(axes.flat, test_ids[:16]):
    with Image.open(TEST_DIR / f"{img_id}.png") as im:
        ax.imshow(im.convert("L"), cmap="gray")
    ax.set_title(f"{img_id} → {predictions[img_id]}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()